## Symmetric high pass filter, and its application to complex IQ signals

In [1]:
import numpy as np
from scipy import signal
from numpy.polynomial.polynomial import polyval
import matplotlib.pyplot as plt
import scipy.fft as spfft

In [2]:
# Interactive plotting.  Comment out this next line if inline plots are desired.
%matplotlib qt

In [3]:
# Function to create IQ representation of sine wave at a given freq for a given sample rate.
#
# Inputs: 
#  freq - frequency of sine wave, Hz
#  amp  - amplitude, arbitrary units
#  fs   - sampling rate of sine wave, Hz
#  no_samps - number of samples to generate
#
# Returns:
#  complex (IQ) representation of sine wave with input parameters.
#
# Affects: None
#
# Exceptions: AssertionError if freq >= fs.
#
def create_sig(freq, amp, fs, no_samps):
    assert freq < fs
    delta_t = 1.0 / fs
    x = np.linspace(0.0, no_samps * delta_t, no_samps, endpoint=False)
    iq = amp * np.exp(1j * freq * 2.0 * np.pi * x)
    return iq

In [4]:
# Function to plot the frequency domain spectrum of a complex signal.
#
# Inputs: 
#  y - complex time domain signal to be plotted
#  fs - sampling rate, Hz
#  ttext - title of plot
#  xlim - x axis plot limits: (min, max)
#  ylim - y axis plot limits: (min, max)
#
# Returns:
#  Plot of frequency domain representation of signal.
#
# Affects: None
#
# Exceptions: None
#
def spec_plot(y, fs, ttext, xlim, ylim):
    delta_t = 1.0 / fs
    no_samps = len(y)
    yf = spfft.fft(y)
    xf = spfft.fftfreq(no_samps, delta_t)
    xf_shift = spfft.fftshift(xf)
    yf_shift = spfft.fftshift(yf)
    plt.figure()
    plt.plot(xf_shift, 1.0/no_samps * np.abs(yf_shift))
    plt.xlim(xlim)
    plt.ylim(ylim)
    plt.xlabel('Frequency, Hz')
    plt.ylabel('Spectral amplitude')
    plt.title(ttext)
    plt.grid()
    plt.show()

In [5]:
# Function to design a symmetric high pass finite impulse response (FIR) filter design 
#  with linear phase response, using the window method with "firwin".
#  The filter is intended to be used with baseband complex signals.
#
# A symmetric high pass filter passes all frequencies < f_cutoff and > f_cutoff 
# and therefore removes all frequencies between -f_cutoff and f_cutoff.
#
# Inputs: 
#  bandwidth - Frequency above which all signals are passed,
#              defined at the half-amplitude point (attenuation -6 dB).
#  fs - Sampling rate, Hz
#
# Returns:
#  h_lp - FIR filter coefficients as array with 251 tap entries.
#
# Affects: None
#
# Exceptions: None
#
def highpass_filt_def(bandwidth, fs):
    
    # This is a filter to be used on a complex baseband signal, so high pass filter cutoff
    #  is set to half the desired bandwidth.
    f_hp_cutoff = bandwidth / 2.0

    # fixed number of taps for filter order
    num_taps = 251 # Filter order (must be odd for linear phase, which helps in design)

    # design lowpass filter
    h_hp = signal.firwin(num_taps, f_hp_cutoff, fs=fs, pass_zero='highpass')

    # Filter is complete    
    return h_hp

In [6]:
# Function to calculate filter frequency response of a FIR filter.
#  Note: this is for visualization purposes only - the time domain filter coefficients
#   do the work here for the actual filtering further down, through the "lfilter" operator.
#
# Inputs: 
#  b - numerator coefficients of filter
#  sample_rate   - Sampling rate of filter, Hz
#  
# Returns:
#  freqs - Frequencies at which filter response was computed, Hz
#      range: [-sample_rate/2, +sample_rate/2)
#  H - Filter frequency response, as complex numbers
#
# Affects: None
#
def comp_fil_response(b, sample_rate):
    
    # Directly compute filter response.
    #
    # For more background on why the frequencies are first computed as radians on (-pi, pi] range, 
    # consult a DSP textbook for an explanation of z-transforms - e.g.
    #    Oppenheim & Schafer, Discrete-Time Signal Processing (3rd ed.)
    #    Proakis & Manolakis, Digital Signal Processing: Principles, Algorithms, and Applications
    #    Lyons, Understanding Digital Signal Processing
    #
    
    # Reverse the coefficients due to library incompatibilities:
    #
    # At present, "polyval" requires ordering of numerator coefficients 
    # from the lowest degree (i.e. the constant term) up to the highest degree.
    # However, the filtering operator lfilter() expect the numerator and denominator 
    # in highest to lowest degree form.
    #
    # This is true for numpy version 1.22.2, the version used at the time of writing,
    #   and for scipy version 1.8.0.  It may change in the future; beware.
    #  
    b = b[::-1]

    # (no numerator coefficients because for a FIR filter, the denominator is a constant = 1)
    
    # compute at 2048 frequencies for good fidelity
    w = np.linspace(-np.pi, np.pi, 2048, endpoint=False)
    
    # main loop
    z = np.exp(1j * w)
    num = np.zeros(len(z), dtype=np.complex64)
    for ii in range(len(z)):
        num[ii] = polyval(z[ii], b)
    H = num   # no denominator term = 1

    # convert frequencies to Hz rather than radians
    freqs = w * sample_rate / (2 * np.pi) # convert from radians to freq

    return freqs, H

In [7]:
# Function to plot the frequency domain response of a filter.
#
# Inputs: 
#  w - array of frequencies at which filter response h was computed.
#  H - array of frequency response, as complex numbers.
#  ttext - title of plot
#  xlim - x axis plot limits: (min, max)
#  ylim - y axis plot limits: (min, max)
#
# Returns:
#  Plot of abs(filter response) vs. real frequency.
#
# Affects: None
#
# Exceptions: None
#
def plot_filt(w, H, ttext, xlim, ylim):
    plt.figure()
    plt.plot(np.real(w), np.abs(H))
    plt.ylim(ylim)
    plt.xlim(xlim)
    plt.xlabel("Frequency, Hz")
    plt.ylabel("Magnitude")
    plt.title(ttext)
    plt.grid()
    plt.show()

In [8]:
fs = 32000       # sample rate
no_samps = 32000 # number of samples
cf = 10000       # center frequenccy

In [9]:
# 4 signal frequencies with different amplitudes
sig_1_f = 8000
sig_1_a = 0.7
sig_2_f = 9000
sig_2_a = 0.8
sig_3_f = 11000
sig_3_a = 0.9
sig_4_f = 12000
sig_4_a = 1.0

In [10]:
# shift frequencies from center frequency to baseband
b_sig_1_f = 8000 - cf
b_sig_1_a = 0.7
b_sig_2_f = 9000 - cf
b_sig_2_a = 0.8
b_sig_3_f = 11000 - cf
b_sig_3_a = 0.9
b_sig_4_f = 12000 - cf
b_sig_4_a = 1.0

In [11]:
# create signals individually
b_sig_1 = create_sig(b_sig_1_f, sig_1_a, fs, no_samps)
b_sig_2 = create_sig(b_sig_2_f, sig_2_a, fs, no_samps)
b_sig_3 = create_sig(b_sig_3_f, sig_3_a, fs, no_samps)
b_sig_4 = create_sig(b_sig_4_f, sig_4_a, fs, no_samps)
# sum them to create input complex signal
b_sig = b_sig_1 + b_sig_2 + b_sig_3 + b_sig_4

In [12]:
# plot 400 samples of time series before filter
plt.figure()
plt.plot(np.real(b_sig[0:400]),label='I')
plt.plot(np.imag(b_sig[0:400]),label='Q')
plt.legend()
plt.grid()
plt.xlabel("Sample Number")
plt.ylabel("Amplitude")
plt.title('Input complex baseband IQ signal')
plt.show()

In [13]:
# Frequency plot limits will be the same for all plots following this point
ylim = (0, 1.05)
xlim = (-3000, 3000)

In [14]:
# plot spectrum of four baseband signals
spec_plot(b_sig, fs, 
          'Frequency spectrum of signal with [%.0f,%.0f,%.0f,%.0f] Hz' % (b_sig_1_f, b_sig_2_f, b_sig_3_f, b_sig_4_f), 
          xlim, ylim)

In [15]:
# construct the symmetric high pass filter
h_hp = highpass_filt_def(3000, fs)

In [16]:
# calculate filter's frequency response
freqs, H = comp_fil_response(h_hp, fs)

In [17]:
# plot frequency domain filter response
plot_filt(freqs, H, 
          'Symmetric HP filter frequency response [%i frequency steps]' % (len(freqs)), 
          xlim, ylim)

In [18]:
print('Filter response coefficients in time domain:')
print('Numerator: ' + str(h_hp))
print('FIR, so Denominator = 1')

Filter response coefficients in time domain:
Numerator: [ 1.57498081e-04  1.90099729e-04  2.07558808e-04  2.08090124e-04
  1.91017471e-04  1.56877779e-04  1.07471194e-04  4.58441393e-05
 -2.38075911e-05 -9.63262074e-05 -1.65874892e-04 -2.26313397e-04
 -2.71652969e-04 -2.96565759e-04 -2.96912527e-04 -2.70242886e-04
 -2.16216141e-04 -1.36889487e-04 -3.68248243e-05  7.70236818e-05
  1.95662340e-04  3.08650213e-04  4.04905202e-04  4.73672477e-04
  5.05580071e-04  4.93687651e-04  4.34423299e-04  3.28301634e-04
  1.80326218e-04 -8.39483043e-18 -1.99101268e-04 -4.00198484e-04
 -5.84583233e-04 -7.33213300e-04 -8.28505611e-04 -8.56170091e-04
 -8.06908126e-04 -6.77795448e-04 -4.73183144e-04 -2.04982384e-04
  1.07753217e-04  4.39972454e-04  7.62454207e-04  1.04420261e-03
  1.25522873e-03  1.36948418e-03  1.36767726e-03  1.23969057e-03
  9.86333966e-04  6.20209054e-04  1.65528369e-04 -3.43179732e-04
 -8.63448171e-04 -1.34819192e-03 -1.74973318e-03 -2.02420603e-03
 -2.13595460e-03 -2.06150979e-03 -

In [19]:
# apply symmetric HP filter to signal
a = [1.0]  # set so filter is a FIR and not IIR
y = signal.lfilter(h_hp, a, b_sig)

In [20]:
# plot 400 samples of time series after filter
plt.figure()
plt.plot(np.real(y[0:400]),label='I')
plt.plot(np.imag(y[0:400]),label='Q')
plt.legend()
plt.grid()
plt.xlabel("Sample Number")
plt.ylabel("Amplitude")
plt.title('Symmetric HP filtered complex baseband IQ signal')
plt.show()

In [21]:
# plot spectrum of filtered output
spec_plot(y, fs, 
          'Freq. spec. of signal after symmetric HP filter',
          xlim, ylim)